# Notebook 1 
### για προετοιμασία της οντολογίας (merge tbox του OpenPVSignal με διάφορα abox-signals του) 

mOWl = python library για χρήση μηχανικής μάθησης με οντολογίες 

Αυτό το notebook περιλαμβάνει:
* την προετοιμασία της *βασικής* οντολογίας που θα χρησιμοποιηθεί: από το OpenPVSignal το σχήμα και όλα τα σήματα (ABox)
* διόρθωση ασυνεπειων σε αυτή που βγάζουν πρόβλημα στον reasoner
* δημιουργία *ενοποιημένης* οντολογίας για οντότητες με ίδιους κωδικούς αναγνώρισης (πχ drugs με ιδιο ATC classification και διαφορετικό prefix) 

Έλεγχος ότι το notebook τρέχει με σωστές εκδόσεις python, java. Επομένως πρέπει να εμφανιστούν:
* JDK έκδοση >= 22.x.x για τη java
* Python έκοδση μια από: 3.10.x, 3.11.x, 3.12.x

In [ ]:
!java --version
!python --version

* Η mOWL για να χρησιμοποιήσει κώδικα java απαιτεί να ξεκινήσει το JVM
* επειδή είναι μεγάλη οντολογία, μπορεί να έχει θέμα αργότερα με το μέγιστο βάθος στοιχείων => κατάργηση του (Djdk.xml.maxElementDepth=0)
 * αφού φορτωθεί και γίνει αυτή η ρύθμιση για το jvm => load mowl + αρχικοποίηση jvm (mowl.init_jvm)

In [ ]:
import os

os.environ['_JAVA_OPTIONS'] = '-Djdk.xml.maxElementDepth=0'
import mowl
mowl.init_jvm("5g")

from mowl.owlapi import OWLAPIAdapter
from org.semanticweb.owlapi.model import IRI
from java.util import HashSet,ArrayList

iriPrefix = "http://purl.org/OpenPVSignal/OpenPVSignal.owl" #το prefix που όλα τα αξιώματα της οντολογίας θα έχουν
print(iriPrefix)

adapter = OWLAPIAdapter() # για πρόσβαση στο OWL API
manager = adapter.owl_manager # δημιουργία ontology manager
factory = adapter.data_factory # δημιουργία data factory (για δημιουργία axioms, classes κτλ)

filenameOntologyBase='data/ontologyBase.owl'
filenameOntologyUnified = 'data/ontologyUnified.owl'

συνάρτηση για απόθηκευση οντολογίας: σώζει με συγκεκριμένο format + iri οντολογίας

In [ ]:
from mowl.owlapi import OWLAPIAdapter
from java.io import File
from org.semanticweb.owlapi.formats import FunctionalSyntaxDocumentFormat,RDFXMLDocumentFormat

def saveOntology(ontology, filename,saveFormat):
    """
    ontology: η οντολογία που θα αποθηκευτεί
    filename: path+όνομα αρχείου για το που θα αποθηκευτεί
    saveFormat: το format της οντολογιας (πχ functional μαζι με τα prefix)
    save οντολογία στο filename με πχ functional syntax
    """
    file = File(filename)
    manager.setOntologyFormat(ontology, saveFormat)
    manager.saveOntology(ontology, saveFormat, IRI.create(file.toURI()))
    print(f"Saved {ontology.getAxiomCount()} axioms to {filename}")

## Δημιουργία *base* οντολογίας
* για merge το TBox (OpenPVSignal.ttl) + signals του (ABox πχ X2019_m_vortioxetine.and.aggression.2019.5th.ttl)
  * κατά τη δημιουργία της εννιαίας οντολογίας για να διατηρούνται τα prefix από το OpenPVSignal => σωστή χρήση format

In [ ]:
#from org.semanticweb.owlapi.formats import PrefixDocumentFormat
import os
import re
PRINT = 1

def addAxiomsPrefix(filename,prefixDefault,ontology,saveFormat):
    """
    συνάρτηση που φορτώνει την οντολογία από το filename 
    βάζει τα axioms της στην οντολογία και
    κρατά στο format όσα prefix είχε για να τα σώσει και αυτά, και για το default το αντικαταστά με το prefixDefault
    """
    oldOntologyOrKG = manager.loadOntologyFromOntologyDocument(File(filename)) #load αξιώματα του ενός σήματος ή σχ΄ήματος
    manager.addAxioms(ontology,oldOntologyOrKG.getAxioms()) # εισαγωγή τους στην ενιαία οντολογία 
    mapPref= manager.getOntologyFormat(oldOntologyOrKG).getPrefixName2PrefixMap() #load prefix από παλιό
    saveFormat.copyPrefixesFrom(mapPref) #στο καινούριο τα prefix
    saveFormat.setPrefix(prefixDefault,manager.getOntologyFormat(oldOntologyOrKG).getDefaultPrefix())#αντικατάσταση default prefix παλιάς με prefixDefault
    return saveFormat
    
def generatePrefixFromFilename(filename):
    """
    παίρνει το όνομα αρχειού filename πχ X2019_m_vortioxetine.and.aggression.2019.5th.ttl
    φτιάχνει κατανοητό και έγκυρο prefix πχ X2019_m_vortioxetine_and_aggression_2019_5th
    """
    #αφαιρεί την κατάληξη του αρχείου, κρατά μόνο αριθμούς και γράμματα και τα υπόλοιπα σύμβολα σαν τελείες τα μετατρέπει σε _
    return re.sub(r'[^a-zA-Z0-9]', '_',(filename.replace(".ttl","")))

ontology = manager.createOntology() #δημιουργία οντολογίας
# για να σώσει και prefix των σημάτων-σχήματος στην τελική οντολογία θέλει να γίνει format για την σύνταξη. Έστω σε functional syntax:
saveFormat = FunctionalSyntaxDocumentFormat() 
folderPath = "data/fromOpenPVSignal"

# προσθήκη σχήματος
saveFormat=addAxiomsPrefix(os.path.join(folderPath,"OpenPVSignal.ttl"),"",ontology,saveFormat)
print("finish adding axioms + prefixes from TBox-RBox")

# αυτοματοποιήμενη προσθήκη όλων των ABox σημάτων στην οντολογία στη μνήμη
for file in os.listdir(folderPath):
    if file.endswith('.ttl') and file != "OpenPVSignal.ttl": #για κάθε αρχείο που όντως σήμα (και όχι το σχήμα)
        newDefaultPrefix = generatePrefixFromFilename(file) #παραγωγή του prefix για αντικατάσταση του default του σήματος
        saveFormat=addAxiomsPrefix(os.path.join(folderPath,file),newDefaultPrefix,ontology,saveFormat) # προσθήκη στην οντολογία των αξιωμάτων + prefixes
        if PRINT: print(f"added {file} with prefix {newDefaultPrefix}")

# επαναφορά ως default prefix της οντολογίας αυτού του σχήματος
saveFormat.setPrefix("",manager.getOntologyFormat(manager.loadOntologyFromOntologyDocument(File(folderPath+"/OpenPVSignal.ttl"))).getDefaultPrefix())

# αποθήκευση οντολογίας από την μνήμη
saveOntology(ontology,filenameOntologyBase,saveFormat)

## Επίλυση ασυνεπειων για να τρέξει reasoner
Υπάρχουν ασυνέπειες που εντοπίζονται από τον reasoner. Μάλιστα ανάλογα τον reasoner, αυτός εντοπίζει κάποιες φορές παραπάων (πχ pellet φάνηκε πιο αυστηρός σε datatypes ελέγχους). Οπότε διορθώθηκαν:  
* Enumerations->προσθήκη τιμών που λείπουν από τα DataRanges βάση protege (addInEnumeration)
*  Αφαίρεση των αξιωμάτων της υπο-ιδιότητας που προκαλούν ασυνέεια (removeSubPropertyOfAxiom)
* θέματα σε κάποιες Data Property Assertions -> πχ πρέπει να είναι boolean αντί απλό string (fixDatatype)

In [ ]:
from org.semanticweb.owlapi.vocab import OWL2Datatype
from org.semanticweb.owlapi.model import AxiomType
PRINT = 1

def addInEnumeration(ontology,dp,addToRange):
    ranges = ontology.getDataPropertyRangeAxioms(dp)
    for rangeAxiom in ranges:
        dataRange = rangeAxiom.getRange()
        if dataRange.getDataRangeType().name() == "DATA_ONE_OF":
            values = HashSet(dataRange.getValues())
            values.add(factory.getOWLLiteral(addToRange))
            newDataRange = factory.getOWLDataOneOf(values)
            newAxiom = factory.getOWLDataPropertyRangeAxiom(dp, newDataRange)
            remove.add(rangeAxiom)
            add.add(newAxiom)
            
def removeSubPropertyOfAxiom(ontology,dp,roleNameString):     
    # για κάθε SubProperty axiom με refers_to_dose_value την υπο-ιδιότητα (ή has_reporter_type το άλλο)
    for axiom in ontology.getDataSubPropertyAxiomsForSubProperty(dp): 
        superProperty = axiom.getSuperProperty()
        
        # αν η υπερ-ιδιότητα είναι refers_to_form_of_intake (ή has_registration_date το άλλο)
        if superProperty.isOWLDataProperty() and superProperty.asOWLDataProperty().getIRI().getShortForm() == roleNameString:
            remove.add(axiom) # για αφαίρεση της
            if PRINT: print(f"removed subproperty that caused problems: {axiom}")
def fixDatatype(condition,literal,newDatatype):
        if condition:
            value = literal.getLiteral()
            newLiteral = factory.getOWLLiteral(value, newDatatype) # double literal σε xsd:decimal
            remove.add(axiom)
            add.add(factory.getOWLDataPropertyAssertionAxiom(dp, subject, newLiteral))
            if PRINT: print(f"changed datatype to {newLiteral}")
#------------------------------------------------------------------------------------------------------------------------------------#
remove = HashSet() # set axioms που θα αφαιρεθούν
add = HashSet() # set axioms που θα μπουν 
datatypeDate = factory.getOWLDatatype(IRI.create("http://www.w3.org/2001/XMLSchema#date"))

for dp in ontology.getDataPropertiesInSignature(): # για όλες τις datatype
    dpRoleName = dp.getIRI().getShortForm()
    
     # 1) Διόρθωση Enumerations->προσθήκη τιμών που λείπουν από τα DataRanges βάση protege 

    # Fix: Προσθήκη του "mg/kg" στο has_dosage_unit
    if dpRoleName == "has_dosage_unit":
        addInEnumeration(ontology,dp,"mg/kg")
    # Fix: Προσθήκη του "death after drug withdrawl" στο refers_to_outcome_after_action
    elif dpRoleName == "refers_to_outcome_after_action":
        addInEnumeration(ontology,dp,"death after drug withdrawl")
                
    # 2) Αφαίρεση αξιώματος υπο-διότητας εκεί που προκαλεί ασυνέεια
    elif dpRoleName == "refers_to_dose_value": #προβληματικό property
        removeSubPropertyOfAxiom(ontology,dp,"refers_to_form_of_intake")

    # αυτο το βρηκε ο pellet οχι ο hermit:
    elif dpRoleName == "has_reporter_type": #προβληματικό property
        removeSubPropertyOfAxiom(ontology,dp,"has_registration_date")

# 3) Διόρθωση των Data Property Assertions (πχ Boolean mismatches από protege)
for axiom in ontology.getAxioms(AxiomType.DATA_PROPERTY_ASSERTION):
    dp = axiom.getProperty().asOWLDataProperty()
    subject = axiom.getSubject()
    literal = axiom.getObject()
    dpRoleName = dp.getIRI().getShortForm()

    # Fix: "refers_to_dechallenge_process" πρέπει να είναι xsd:boolean αντι για απλό string "true"
    if dpRoleName == "refers_to_dechallenge_process":
        value = (str(literal.getLiteral())).lower() # ολα μικρα για να βρει πχ True,true κτλ
        if value in ["true", "false"] and not literal.getDatatype().isBoolean():
            newLiteral = factory.getOWLLiteral((value == "true")) # δημιουργία σωστού boolean literal
            remove.add(axiom)
            add.add(factory.getOWLDataPropertyAssertionAxiom(dp, subject, newLiteral))

    # Fix: "has_value" πρέπει να είναι xsd:decimal (στο protege εμφανίστηκε ως 'xsd:double "2.0E-4"')
    elif dpRoleName == "has_value":
        fixDatatype(literal.getDatatype().getIRI() == OWL2Datatype.XSD_DOUBLE.getIRI(),literal,OWL2Datatype.XSD_DECIMAL)
        
    # Fix: "refers_to_dechallenge_process" πρέπει να είναι xsd:boolean αντι για απλό string "true"
    elif dpRoleName == "has_creation_date":
        fixDatatype(literal.getDatatype().getIRI() == OWL2Datatype.XSD_INTEGER.getIRI() or literal.getDatatype().getIRI() == OWL2Datatype.XSD_STRING.getIRI(),literal,datatypeDate)

    # Fix: "initially_identified_on" πρέπει να είναι xsd:date αντι για xsd:string    
    elif dpRoleName == "initially_identified_on":
         fixDatatype(literal.getDatatype().getIRI() == OWL2Datatype.XSD_STRING.getIRI(),literal,datatypeDate)
#------------------------------------------------------------------------------------------------------------------------------------#
    
# Εφαρμογή όλων των αλλαγών (αφαιρέσεις + προσθήκες)
manager.removeAxioms(ontology, remove)
manager.addAxioms(ontology, add)

# Αποθήκευση
saveOntology(ontology, filenameOntologyBase, saveFormat)
print("Ontology fixed and saved successfully!")

fix Dates:
* πχ το 1/10/2013 το έχει έτσι ενώ το δηλώνει ως xsd:date
   * όμως το xsd:date απαιτεί αυστηρό format YYYY-MM-DD (με αυτή τη σειρά και '-' αντί για '/')
   * οπότε remove αυτό και add το ίδιο με σωστό format
* ο pellet φαίνεται πιο αυστηρός από hermit σε έλεγχο datatypes->απαιτεί xsd:date ως YYYY-MM-DD και όταν βλέπει πχ σκέτο "2013" βγάζει error
  * λύση να δωθεί στο range της κάθε τέτοιας ιδιότητας πχ has_creation_date ότι είναι union date+year ή
  * να δωθεί αυθέρετα μήνας-μέρα πχ 01.01 για να μείνει date

In [ ]:
PRINT = 1

from org.semanticweb.owlapi.vocab import OWL2Datatype
from org.semanticweb.owlapi.model import AxiomType
#fix πχ "1/10/2013"^^xsd:date (αντικατάσταση με "2013-10-01"^^xsd:date) ΄ή λάθος μορφ΄ή για dateTime
#το τελευταιο με 2,4 γιατι καποια τα εχουν ως πχ 13 αντι 2013 (τα βρηκε pellet που ηταν string) + για / και - διαχωριστικό
oldDateLiteral = re.compile(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})$") 
datatypeDate = factory.getOWLDatatype(IRI.create("http://www.w3.org/2001/XMLSchema#date"))

remove = HashSet()
add = HashSet()

for axiom in ontology.getAxioms():
    if axiom.getAxiomType() == AxiomType.DATA_PROPERTY_ASSERTION:
        literal = axiom.getObject().getLiteral() 
        datatype = axiom.getObject().getDatatype().getIRI()
        newLiteralString=None

        # αυστηρο format date που χτυπά πχ "1/10/2013"^^xsd:date ή με 13 αντι 2013 ή με - αντι /
        find=oldDateLiteral.match(str(literal))
        if find: 
                day, month, year = find.groups()
                if len(year)==2: year = "20" + year # να ειναι 2013 αντι 13
                newLiteralString = f"{year}-{int(month):02d}-{int(day):02d}" #για σωστή μορφ΄ή για date datatype
        
        # αυστηρο format date για pellet (όχι ελλειπείς)
        if datatype==IRI.create("http://www.w3.org/2001/XMLSchema#date"):
            if re.match(r"^\d{4}$", str(literal)): #"2013"^^xsd:date  => 2013-01-01 αρα βαλε μερα +μηνα
                newLiteralString = literal + '-01-01'
            elif re.match(r"^\d{4}-\d{2}$", str(literal)): #"2013-01"^^xsd:Date =>2013-01-01 αρα βαλε μερα μονο
                newLiteralString = literal + '-01'
                
        if newLiteralString:  # αν αλλαξε=>ενημερωσε
            subject = axiom.getSubject()
            role = axiom.getProperty()
            newLiteral = factory.getOWLLiteral(newLiteralString,datatypeDate) #νέο literal με τη σωστή τιμή + datatype date
          
            newAxiom = factory.getOWLDataPropertyAssertionAxiom(role,subject,newLiteral) # το αξίωμα με το σωστό literal που θα μπει
            add.add(newAxiom)
            remove.add(axiom) #για να αφαιρεθεί το παλιό
            if PRINT: print(f"changed {literal} to {newLiteralString}")
                
manager.removeAxioms(ontology, remove)
manager.addAxioms(ontology,add)
saveOntology(ontology,filenameOntologyBase,saveFormat)

## Δημιουργία *unified* οντολογίας
Όταν διαφορετικές οντότητες μοιράζονται τον ίδιο κωδικό αναγνώρισης (πχ ATC classification) στη *βασική* οντολογία είχαν καταγραφεί ως μη συνδεδεμένοι κόμβοι, με strings (και οι ίδιες οι οντότητες με διαφορετικά prefix).

Δηλαδή άτομα με άλλο prefix και ίδιους κωδικούς αναγνώρισης επειδή είναι σε άλλα σήματα για:
* drugs με has_ATC_code
* adverse effects με has_MedDRA_code
* Indication με has_ICD_code
   * αν και συνηθως adverse effect ταυτοχρονα ή και μονο indication σε αυτά
   * πχ donepezilInteractionSeretoninSyndrome2012:fever, δηλαδή για 1 αλληλεπίδραση του drug donepezil που προκάλεσε Seretonin Syndrome
   * ο πυρετος είναι παρενεργεια (Adverse Effect) για drug donepezil εδώ
   * ομώς μπορεί για άλλο drug (πχ ντεπον) πυρετός να είναι ενδειξη για χορηγηση drug
   * άρα δήλωση depon ως indication + adverse effect

Θεμα αν δεν συνδέονται κάπως στη μηχανική μάθηση γιατί για πχ signal1:fever + signal2:fever θα τα θεωρεί 2 άσχετα άτομα 
* μετά δεν θα δει ότι τα 2 σήματα μοιράζονται τον κωδικό αυτό, οπότε πιθανώς θα αποτύχει να τα συνδέσει και να βρει μοτίβα μεταξύ τους
 
Εδώ υπάρχουν διάφορες λύσεις με θέματα:
* να μπει στη θέση κάθε πχ signali:fever ίδιο prefix για να εχουν ίδιο IRI
    * τότε πληροφορίες όπως ότι ο πυρετός σε 1 σήμα συνδέεται με συγκεκριμένες πχ ηλικίες θα χαθεί
*  Entity Alignment
   * να δηλωθεί ότι αν τα άτομα άτομα με ίδιο κωδικό αναγνώρισης (πχ meddra_10039020) είναι sameAs
   * αν δεν τρέξει reasoner δεν θα το καταλάβει η οντολογία
   * αν τρέξει ο reasoner τότε θα προσθέσει τα κατάλληλα αξιώματα και πάλι θα χαθεί πληροφορία (με ποιανού σήματος πληροφορίες συνδέονταν το κάθε άτομο πριν την δήλωση sameAs)

Τελικά επιλέχθηκε:
*  μετατροπή κάθε μοναδικού κωδικού αναγνώρισης για αυτά τα 3 σε node (νέο άτομο) και χρηση 1 ιδιότητας για να συνδέσουν 2 άτομα με ίδιο τέτοιο κωδικο με τον νέο κόμβο
* πχ signal1:fever και signal2fever ίδιο medra code '10037660'=>node medDRA__10037660 και με πχ ρόλο has_MedDRA_Concept σύνδεση των 2 ατόμων πυρετού με το medDRA__10037660

In [ ]:
from IPython.display import HTML
HTML("""
<div style="display: flex; gap: 10px;">
    <img src="imagesNotebook/unified.drawio.png" width="1000" />
</div>
""")

In [ ]:
PRINT = 0
SHOW_ALL_AXIOMS_ADD =0

def makeNewAxioms(sameCode,newAxioms,className,propertyName,indPrefix,comment):
    '''
    δημιουργεί για κάθε κωδικό αναγνώρισης(που ήταν σε string) ένα άτομο πχ meddra_10039020 για το '10039020' και 
    μια νέα ιδιότητα που συνδέει μια οντότητα με το νέο άτομο αυτό
    για το range της ιδιότητας μια κλάση ανάλογα τι κωδικός ήταν (πχ για φάρμακα)
    
    μετά εντοπίζει οντότητες (φάρμακα, παρενέργειες, παθήσεις) που μοιράζονται ίδιο κωδικό (annotation property) και 
    τις συνδέει με τα κάταλληλα νέα άτομα
    Args:
        sameCode (str): string που περιέχεται στο IRI του κωδικού αναγνώρισης (πχ has_MedDRA_code)
        newAxioms (HashSet): set όπου θα προστεθούν τα νέα OWL axioms
        className (str): όνομα της νέας κλάσης που δημιουργείται για αυτούς τους κωδικούς (πχ #MedDRA_Concept)
        propertyName (str): νέα Object Property που θα συνδέει την οντότητα με τον νέο κόμβο ατόμου(πχ #has_MedDRA_Concept)
        indPrefix (str): πρόθεμα για το IRI του νέου ατόμου-κωδικού (πχ #meddra_)
        comment (str): rdfs:comment για τη νέα Object Property
    Returns:
        HashSet: το ενημερωμένο set με τα αξιώματα
    '''
    # πχ :MedDRA_Concept ωστε το ατομο :meddra_10039020 να ειναι μελος της
    conceptClass = factory.getOWLClass(IRI.create(iriPrefix + className)) 
    # πχ :has_MedDRA_Concept ωστε τα adverse effect με κωδικο 10039020 να ενωνονται με το ατομο meddra_10039020
    hasConceptProperty = factory.getOWLObjectProperty(IRI.create(iriPrefix + propertyName))
    #axioms για declaration νεας κλασης-ιδιοτητας
    newAxioms.add(factory.getOWLDeclarationAxiom(conceptClass))
    newAxioms.add(factory.getOWLDeclarationAxiom(hasConceptProperty))
    #axioms για range νεας ιδιοτητας + rdfs:comment για το τι ειναι 
    newAxioms.add(factory.getOWLObjectPropertyRangeAxiom(hasConceptProperty,conceptClass))
    newAxioms.add(factory.getOWLAnnotationAssertionAxiom(factory.getRDFSComment(),hasConceptProperty.getIRI(),factory.getOWLLiteral(comment)))
                  
    sameCodeDictionary = {}
    # collect σε 1 λεξικο τα τους κωδικους αναγνώρισης (key) με τα ατομα που τα εχουν (λιστα) πχ για κωδικο 10039020 τα [adverseEffect1,adverseEffect2,...]
    for axiom in ontology.getAxioms():
        if axiom.getAxiomType() == AxiomType.ANNOTATION_ASSERTION :
            if sameCode in str(axiom.getProperty().toStringID()):
                entity, code = axiom.getSubject(),axiom.getAnnotation().annotationValue().getLiteral()
                #print(entity,code)
                if code not in sameCodeDictionary: sameCodeDictionary[code] = HashSet()
                sameCodeDictionary[code].add(factory.getOWLNamedIndividual(entity))
   
    # δημιουργια νεου node για καθε key (πχ για meddra code '1' το iri+meddra_1) + συνδεση του με τα ατομα που ειχαν τον κωδικο αυτον 
    for code in sameCodeDictionary: 
        if PRINT: print('\n',code)
        #πχ για meddra 10039020 νεο ατομο :meddra_10039020 (+axiom για το declaration του)
        conceptIndividual = factory.getOWLNamedIndividual(IRI.create(iriPrefix + indPrefix + str(code)))
        newAxioms.add(factory.getOWLDeclarationAxiom(conceptIndividual))
        if PRINT: print(factory.getOWLClassAssertionAxiom(conceptClass,conceptIndividual))
        # axiom: meddra_10039020 type :MedDRA_Concept
        newAxioms.add(factory.getOWLClassAssertionAxiom(conceptClass,conceptIndividual))
        # axiom: adverseEffect1 :has_MedDRA_Concept :meddra_10039020 οπου το adverseEffect1 απο sameCodeDictionary καθε ατομο λιστας με key=code
        for entity in sameCodeDictionary[code]: 
            if PRINT:    print(entity)
            newAxioms.add(factory.getOWLObjectPropertyAssertionAxiom(hasConceptProperty,entity,conceptIndividual))
            if PRINT: print(factory.getOWLObjectPropertyAssertionAxiom(hasConceptProperty,entity,conceptIndividual))
    if SHOW_ALL_AXIOMS_ADD: # print όλα τα νέα αξιώματα
        for axiom in newAxioms: 
            print(axiom)
            print()
    return newAxioms
    
newAxioms = HashSet()
print('-'*30,'adverse effects with same MedDRA code:','-'*30)
makeNewAxioms('has_MedDRA_code',newAxioms,"#MedDRA_Concept","#has_MedDRA_Concept",'#medDRA_','connects an adverse effect individual with the new medDRA individual based on medDRA code')
print('-'*30,'conditions with same has_ICD_code code:','-'*30)
makeNewAxioms('has_ICD_code',newAxioms,"#ICD_Concept","#has_ICD_Concept",'#icd_','connects an entity with the new ICD individual based on ICD code')
print('-'*30,'drugs with same  has_ATC_code code:','-'*30)
makeNewAxioms('has_ATC_code',newAxioms,"#ATC_Concept","#has_ATC_Concept",'#atc_','connects a drug with the new ATC individual based on ATC code')

ontologyUnified = manager.createOntology()
manager.addAxioms(ontologyUnified,ontology.getAxioms())
manager.addAxioms(ontologyUnified, newAxioms)

saveOntology(ontologyUnified,filenameOntologyUnified,saveFormat)

### έλεγχος συνέπειας
αν οι δύο εκδόσεις των οντολογιών (με τις διορθώσεις) είναι συνεπείς βάση HermiT

In [ ]:
from org.semanticweb.HermiT import Reasoner

reasoner = Reasoner.ReasonerFactory().createReasoner(ontology) # αρχικοποιήση reasoner
print("is base ontology consistent according to Hermit: ",reasoner.isConsistent())
reasoner = Reasoner.ReasonerFactory().createReasoner(ontologyUnified) # αρχικοποιήση reasoner
print("is unified ontology consistent according to Hermit: ",reasoner.isConsistent())